# 🔗 Core Bridge

> Zero-copy connection between PyArrow and MAX Engine.
>
> **Path**: Arrow → NumPy view → MAX Tensor *(no bytes copied on CPU)*

This module provides the foundational data bridge for mxframe. All data flows through this zero-copy pipeline to enable GPU-accelerated DataFrame operations without memory overhead.

In [ ]:
#| default_exp core_bridge

In [ ]:
#| export
import pyarrow as pa
import numpy as np
from typing import Dict, Optional, Union, Literal
from max import driver
from max.dtype import DType
import datetime
import time
from nbdev.showdoc import show_doc

In [ ]:
#| export
ARROW_TO_MAX_DTYPE: Dict[pa.DataType, DType] = {
    pa.int8(): DType.int8,
    pa.int16(): DType.int16,
    pa.int32(): DType.int32,
    pa.int64(): DType.int64,
    pa.uint8(): DType.uint8,
    pa.uint16(): DType.uint16,
    pa.uint32(): DType.uint32,
    pa.uint64(): DType.uint64,
    pa.float32(): DType.float32,
    pa.float64(): DType.float64,
    pa.date32(): DType.int32,
}

ARROW_TO_NUMPY_DTYPE: Dict[pa.DataType, np.dtype] = {
    pa.int8(): np.dtype('int8'),
    pa.int16(): np.dtype('int16'),
    pa.int32(): np.dtype('int32'),
    pa.int64(): np.dtype('int64'),
    pa.uint8(): np.dtype('uint8'),
    pa.uint16(): np.dtype('uint16'),
    pa.uint32(): np.dtype('uint32'),
    pa.uint64(): np.dtype('uint64'),
    pa.float32(): np.dtype('float32'),
    pa.float64(): np.dtype('float64'),
    pa.date32(): np.dtype('int32'),
}

def get_max_dtype(arrow_type: pa.DataType) -> DType:
    """Get MAX DType for an Arrow type."""
    if arrow_type in ARROW_TO_MAX_DTYPE:
        return ARROW_TO_MAX_DTYPE[arrow_type]
    raise TypeError(f"Unsupported Arrow type: {arrow_type}")


def get_numpy_dtype(arrow_type: pa.DataType) -> np.dtype:
    """Get NumPy dtype for an Arrow type."""
    if arrow_type in ARROW_TO_NUMPY_DTYPE:
        return ARROW_TO_NUMPY_DTYPE[arrow_type]
    raise TypeError(f"Unsupported Arrow type: {arrow_type}")

In [ ]:
show_doc(get_numpy_dtype)

---

### get_numpy_dtype

```python

def get_numpy_dtype(
    arrow_type:DataType
)->dtype:


```

*Get NumPy dtype for an Arrow type.*

Unsupported types raise `TypeError`:

In [ ]:
#| eval: false
assert get_max_dtype(pa.float64()) == DType.float64
assert get_max_dtype(pa.date32()) == DType.int32  # Date32 maps to int32

In [ ]:
#| export
def arrow_to_numpy_view(
    arr: Union[pa.Array, pa.ChunkedArray]  # PyArrow array (primitive type, no nulls)
) -> np.ndarray:  # NumPy view over same memory
    """Get zero-copy NumPy view of an Arrow array."""
    # Handle ChunkedArray - combine if needed
    if isinstance(arr, pa.ChunkedArray):
        if arr.num_chunks == 1:
            arr = arr.chunk(0)
        else:
            # Multiple chunks - must combine (may copy)
            arr = arr.combine_chunks()
    
    # Check for nulls
    if arr.null_count > 0:
        raise ValueError(f"Array has {arr.null_count} nulls - zero-copy not possible")
    
    # Try direct zero-copy
    try:
        return arr.to_numpy(zero_copy_only=True)
    except pa.ArrowInvalid:
        # Special handling for Date32 - reinterpret buffer as int32
        if pa.types.is_date32(arr.type):
            data_buffer = arr.buffers()[1]
            return np.frombuffer(data_buffer, dtype=np.int32)
        raise TypeError(f"Cannot create zero-copy view for {arr.type}")

In [ ]:
arr = pa.array([1.0, 2.0, 3.0])
np_view = arrow_to_numpy_view(arr)
assert np_view.ctypes.data == arr.buffers()[1].address  # Zero-copy!

In [ ]:
try:
    arrow_to_numpy_view(pa.array([1.0, None, 3.0]))
    assert False, "Should have raised ValueError"
except ValueError as e:
    assert "nulls" in str(e)

In [ ]:
#| export
def arrow_to_max_tensor(
    arr: Union[pa.Array, pa.ChunkedArray],  # PyArrow array to convert
    device: Optional[driver.Device] = None  # Target device (`None` = CPU)
) -> driver.Tensor:  # MAX Tensor (zero-copy on CPU, copied on GPU)
    """Zero-copy bridge from PyArrow array to MAX Tensor."""
    # Get zero-copy NumPy view
    np_view = arrow_to_numpy_view(arr)
    
    # Create MAX Tensor from NumPy (zero-copy for contiguous arrays)
    tensor = driver.Tensor.from_numpy(np_view)
    
    # Copy to GPU if requested
    if device is not None and not device.is_host:
        return tensor.to(device)
    
    return tensor

In [ ]:
show_doc(arrow_to_max_tensor)

---

### arrow_to_max_tensor

```python

def arrow_to_max_tensor(
    arr:Union, # PyArrow array to convert
    device:Optional=None, # Target device (`None` = CPU)
)->Tensor: # MAX Tensor (zero-copy on CPU, copied on GPU)


```

*Zero-copy bridge from PyArrow array to MAX Tensor.*

In [ ]:
#| export
DeviceType = Literal["cpu", "gpu", "auto"]

class MXFrame:
    """PyArrow-backed DataFrame with zero-copy MAX Engine integration."""
    
    def __init__(self, 
                 data: Union[pa.Table, Dict[str, list]]  # Arrow Table or dict of lists
                ):
        """Create MXFrame from Arrow Table or dict."""
        if isinstance(data, pa.Table):
            self._table = data
        elif isinstance(data, dict):
            self._table = pa.table(data)
        else:
            raise TypeError(f"Expected Table or dict, got {type(data)}")
        
        # Cache for numpy views (keeps references alive)
        self._numpy_cache: Dict[str, np.ndarray] = {}
        
        # Resolved device
        self._device: Optional[driver.Device] = None
    
    @property
    def columns(self) -> list:
        """Column names."""
        return self._table.column_names
    
    @property
    def num_rows(self) -> int:
        """Number of rows."""
        return self._table.num_rows
    
    @property
    def num_columns(self) -> int:
        """Number of columns."""
        return self._table.num_columns
    
    def __len__(self) -> int:
        """Number of rows."""
        return self.num_rows
    
    def __repr__(self) -> str:
        """String representation."""
        return f"MXFrame({self.num_rows} rows, {self.columns})"
    
    def column(self, 
               name: str  # Column name
              ) -> pa.ChunkedArray:  # Arrow column
        """Get Arrow column by name."""
        return self._table.column(name)
    
    def __getitem__(self, name: str) -> pa.ChunkedArray:
        """Get Arrow column by name via `df['col']`."""
        return self.column(name)
    
    def to_arrow(self) -> pa.Table:
        """Get underlying Arrow table."""
        return self._table
    
    # ========== Zero-Copy Bridge Methods ==========
    
    def to_numpy(self, 
                 column: str  # Column name
                ) -> np.ndarray:  # Zero-copy NumPy view
        """Get zero-copy NumPy view of column (cached)."""
        if column not in self._numpy_cache:
            self._numpy_cache[column] = arrow_to_numpy_view(self._table.column(column))
        return self._numpy_cache[column]
    
    def to_max_tensor(self, 
                      column: str,  # Column name
                      device: Optional[driver.Device] = None  # Target device (`None` = CPU)
                     ) -> driver.Tensor:  # MAX Tensor
        """Get MAX Tensor for column (zero-copy on CPU)."""
        return arrow_to_max_tensor(self._table.column(column), device)
    
    def get_buffer_address(self, 
                           column: str  # Column name
                          ) -> int:  # Memory address
        """Get memory address of column's data buffer (for zero-copy verification)."""
        arr = self._table.column(column)
        if isinstance(arr, pa.ChunkedArray):
            arr = arr.chunk(0) if arr.num_chunks == 1 else arr.combine_chunks()
        return arr.buffers()[1].address

In [ ]:
show_doc(MXFrame.to_numpy)

---

### MXFrame.to_numpy

```python

def to_numpy(
    column:str, # Column name
)->ndarray: # Zero-copy NumPy view


```

*Get zero-copy NumPy view of column (cached).*

In [ ]:
df = MXFrame({'x': [1.0, 2.0, 3.0]})
np1 = df.to_numpy('x')
np2 = df.to_numpy('x')
assert np1 is np2  # Same cached view

Pass a `device` to copy to GPU (unavoidable for GPU compute):

In [ ]:
show_doc(MXFrame.get_buffer_address)

---

### MXFrame.get_buffer_address

```python

def get_buffer_address(
    column:str, # Column name
)->int: # Memory address


```

*Get memory address of column's data buffer (for zero-copy verification).*

In [ ]:
df = MXFrame({'x': [1.0, 2.0, 3.0]})
arrow_addr = df.get_buffer_address('x')
numpy_addr = df.to_numpy('x').ctypes.data
assert arrow_addr == numpy_addr  # Same memory!

In [ ]:
#| eval: false
# 📋 Test 1: Create MXFrame
df = MXFrame({
    'price': [10.0, 20.0, 30.0, 40.0, 50.0],
    'qty': [1, 2, 3, 4, 5],
})

print(f"Created: {df}")
print(f"Columns: {df.columns}")
print(f"Rows: {df.num_rows}")

Created: MXFrame(5 rows, ['price', 'qty'])
Columns: ['price', 'qty']
Rows: 5


In [ ]:
#| eval: false
# ⚡ Test 3: Arrow to MAX Tensor
tensor = df.to_max_tensor('price')

print(f"Tensor shape: {tensor.shape}")
print(f"Tensor dtype: {tensor.dtype}")
print(f"Tensor values: {tensor.to_numpy()}")
print(f"Arrow values:  {df['price'].to_pylist()}")

Tensor shape: (5,)
Tensor dtype: DType.float64
Tensor values: [10. 20. 30. 40. 50.]
Arrow values:  [10.0, 20.0, 30.0, 40.0, 50.0]


In [ ]:
#| eval: false
# 🔢 Test 5: Integer column
numpy_qty = df.to_numpy('qty')
tensor_qty = df.to_max_tensor('qty')

print(f"qty NumPy dtype: {numpy_qty.dtype}")
print(f"qty Tensor dtype: {tensor_qty.dtype}")
print(f"qty values: {numpy_qty}")

qty NumPy dtype: int64
qty Tensor dtype: DType.int64
qty values: [1 2 3 4 5]


In [ ]:
#| eval: false
# 🚀 Test 7: Large array performance test (proves zero-copy)

n = 10_000_000
large_arr = pa.array(np.random.rand(n).astype(np.float32))
df_large = MXFrame(pa.table({'data': large_arr}))

# Time the zero-copy conversion
t0 = time.perf_counter()
for _ in range(100):
    tensor = df_large.to_max_tensor('data')
elapsed = (time.perf_counter() - t0) / 100 * 1000

print(f"Array size: {n:,} elements ({n * 4 / 1e6:.1f} MB)")
print(f"Arrow → MAX Tensor: {elapsed:.3f} ms")
print(f"Throughput: {n * 4 / 1e9 / (elapsed / 1000):.1f} GB/s")
print("(Fast time = zero-copy confirmed)")

Array size: 10,000,000 elements (40.0 MB)
Arrow → MAX Tensor: 8.594 ms
Throughput: 4.7 GB/s
(Fast time = zero-copy confirmed)


In [ ]:
#| eval: false
# ❌ Test 9: Error handling - nulls should fail
arr_with_nulls = pa.array([1.0, None, 3.0])
df_nulls = MXFrame(pa.table({'col': arr_with_nulls}))

try:
    _ = df_nulls.to_numpy('col')
    print("ERROR: Should have raised ValueError!")
except ValueError as e:
    print(f"Correctly rejected nulls: {e}")

Correctly rejected nulls: Array has 1 nulls - zero-copy not possible


In [ ]:
#| hide
from nbdev import nbdev_export
nbdev_export()